# Day 26: LLM Orchestration, Tool Calling and LangGraph
**Chains, LangChain v1 agents and middleware, safe tool calling, the Agents SDK and durable LangGraph workflows**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Install

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai tiktoken scikit-learn numpy pandas langchain langchain-core langchain-openai langgraph openai-agents nest_asyncio

# ===== Option B: VS Code / local =====
# pip install openai tiktoken scikit-learn numpy pandas langchain langchain-core langchain-openai langgraph openai-agents nest_asyncio
# Create a .env file (git-ignored) or set the variable in your terminal:  OPENAI_API_KEY=...
# No key? Everything still runs in OFFLINE demo mode with a rule-based stand-in model.
print("Setup done")

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

### A chat model for LangChain and LangGraph
Online you get `ChatOpenAI`. Offline you get a small rule-based model so the agent loop, middleware, checkpoints and interrupts really run.

In [ ]:
%%writefile lc_offline.py
"""Chat model for the LangChain and LangGraph demos.
ONLINE : get_chat_model() returns ChatOpenAI with the program model (LLM_MODEL).
OFFLINE: a small rule-based chat model. It is NOT intelligent. It makes tool calls from simple rules,
         so the agent loop, middleware, checkpoints and interrupts really run without a key.
"""
import itertools, json, re
from typing import Any
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
import genai_helpers

_ids = itertools.count(1)

def _call(name, args):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": f"call_{next(_ids)}"}])

def support_planner(messages, tool_names):
    """Decide the next tool call for the support demos, or None to answer."""
    user = next((m.content for m in reversed(messages) if isinstance(m, HumanMessage)), "")
    low = user.lower()
    used = [m.name for m in messages if isinstance(m, ToolMessage)]
    results = [json.loads(m.content) for m in messages
               if isinstance(m, ToolMessage) and str(m.content).strip().startswith("{")]
    last = results[-1] if results else {}
    oid = re.search(r"ORD-\d{4}", user)
    if oid and "get_order_status" in tool_names and "get_order_status" not in used:
        return _call("get_order_status", {"order_id": oid.group(0)})
    if "refund" in low and "amount" in last and "calculate_refund" in tool_names and "calculate_refund" not in used:
        return _call("calculate_refund", {"amount": last["amount"], "days_late": last.get("days_late", 0)})
    if ("issue" in low or "pay" in low) and "refund" in last and oid and "issue_refund" in tool_names \
            and "issue_refund" not in used:
        return _call("issue_refund", {"order_id": oid.group(0), "amount": last["refund"]})
    if "policy" in low and "search_policy" in tool_names and "search_policy" not in used:
        return _call("search_policy", {"query": user})
    return None

class OfflineChatModel(BaseChatModel):
    tool_names: list = []
    planner: Any = None

    @property
    def _llm_type(self):
        return "offline-demo"

    def bind_tools(self, tools, **kwargs):
        names = []
        for t in tools:
            if hasattr(t, "name"):
                names.append(t.name)
            elif isinstance(t, dict):
                names.append(t.get("function", t).get("name"))
            else:
                names.append(getattr(t, "__name__", str(t)))
        return self.model_copy(update={"tool_names": names})

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        msg = (self.planner or support_planner)(messages, self.tool_names) if self.tool_names else None
        if msg is None:
            facts = [str(m.content) for m in messages if isinstance(m, ToolMessage)]
            human = next((m.content for m in reversed(messages) if isinstance(m, HumanMessage)), "")
            system = next((m.content for m in messages if isinstance(m, SystemMessage)), "")
            text = f"[offline demo] Based on the tools: {facts[-1]}" if facts else genai_helpers.offline_llm(human, system)
            msg = AIMessage(content=text)
        return ChatResult(generations=[ChatGeneration(message=msg)])

def get_chat_model(**kwargs):
    if genai_helpers.ONLINE:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=genai_helpers.MODEL, api_key=genai_helpers.OPENAI_API_KEY, **kwargs)
    return OfflineChatModel(**kwargs)

print("Chat model:", "ChatOpenAI" if genai_helpers.ONLINE else "OfflineChatModel (offline demo)")

In [ ]:
import importlib, lc_offline
importlib.reload(lc_offline)
from lc_offline import get_chat_model
model = get_chat_model()

## 2. Chaining with LCEL
One chain, three audiences, run with `batch()`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

incident = ("The billing service was down for 40 minutes on Monday. 1,200 customers saw a payment error. "
            "A database connection limit was reached; we raised the limit and added an alert.")
prompt = ChatPromptTemplate.from_messages([
    ("system", "You summarise incidents for {audience} in 3 short bullet points."),
    ("user", "{text}")])
chain = prompt | model | StrOutputParser()
for audience, out in zip(["a CIO", "an engineer", "an end user"],
                         chain.batch([{"audience": a, "text": incident} for a in ["a CIO", "an engineer", "an end user"]])):
    print(f"{audience}: {out[:120]}")

## 3. LLM as judge
A second call scores an answer against a rubric. Offline, a simple heuristic stands in.

In [ ]:
import json, re

def judge(question, answer, context):
    if ONLINE:
        r = llm(f"Score the answer from 1 to 5 for relevance and groundedness in the context. "
                f"Reply as JSON with keys score and reason.\nContext: {context}\nQuestion: {question}\nAnswer: {answer}",
                temperature=0)
        try:
            return json.loads(r[r.index("{"): r.rindex("}") + 1])
        except Exception:
            return {"score": None, "reason": "could not parse judge output"}
    words = lambda t: set(re.findall(r"[a-z0-9]+", t.lower()))
    overlap = len(words(answer) & words(context)) / max(1, len(words(answer)))
    return {"score": round(1 + 4 * overlap), "reason": f"offline overlap with context = {overlap:.2f}"}

ctx = "Late deliveries get 5% of the order value per day late, up to 50%."
for ans in ["You get 5% per day late, up to 50%.", "You get a free phone."]:
    print(ans, "->", judge("What refund for late delivery?", ans, ctx))

## 4. Tools: schemas, validation and an allowlist

In [ ]:
from pydantic import BaseModel, Field, ValidationError

ORDERS = {"ORD-1042": {"status": "delivered", "amount": 4800, "days_late": 3},
          "ORD-2077": {"status": "in transit", "amount": 1200, "days_late": 0}}

class OrderArgs(BaseModel):
    order_id: str = Field(pattern=r"^ORD-\d{4}$")

class RefundArgs(BaseModel):
    amount: float = Field(gt=0)
    days_late: int = Field(ge=0)

def get_order_status_fn(order_id):
    return json.dumps(ORDERS.get(order_id, {"error": "order not found"}))

def calculate_refund_fn(amount, days_late):
    return json.dumps({"refund": round(amount * min(0.05 * days_late, 0.5), 2)})   # 5% per day late, max 50%

REGISTRY = {"get_order_status": (get_order_status_fn, OrderArgs), "calculate_refund": (calculate_refund_fn, RefundArgs)}
TOOLS = [
    {"type": "function", "function": {"name": "get_order_status",
        "description": "Look up an order by id, for example ORD-1042. Returns status, amount and days late.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}},
    {"type": "function", "function": {"name": "calculate_refund",
        "description": "Calculate a late delivery refund from the order amount and days late.",
        "parameters": {"type": "object", "properties": {"amount": {"type": "number"}, "days_late": {"type": "integer"}},
                       "required": ["amount", "days_late"]}}},
]

def run_tool(name, args_json):
    if name not in REGISTRY:
        return json.dumps({"error": f"unknown tool {name}"})
    fn, schema = REGISTRY[name]
    try:
        args = schema(**json.loads(args_json))
    except ValidationError as e:
        return json.dumps({"error": f"invalid arguments: {e.errors()[0]['msg']}"})
    except json.JSONDecodeError as e:
        return json.dumps({"error": f"invalid JSON: {e}"})
    return fn(**args.model_dump())

print(run_tool("delete_database", "{}"))
print(run_tool("get_order_status", '{"order_id": "1042"}'))
print(run_tool("calculate_refund", '{"amount": 4800, "days_late": 3}'))

## 5. The tool calling loop with the direct OpenAI API

In [ ]:
import genai_helpers

def offline_tool_model(messages):
    # Stand-in for a model that supports tool calling: decides the next step from what it has seen so far.
    user = messages[0]["content"]
    results = [json.loads(m["content"]) for m in messages if m["role"] == "tool"]
    oid = re.search(r"ORD-\d{4}", user)
    if not results and oid:
        return {"tool": "get_order_status", "args": json.dumps({"order_id": oid.group(0)})}
    if len(results) == 1 and "amount" in results[0]:
        return {"tool": "calculate_refund", "args": json.dumps({"amount": results[0]["amount"], "days_late": results[0]["days_late"]})}
    if results and "refund" in results[-1]:
        return {"answer": f"Your refund is {results[-1]['refund']} INR for the late delivery."}
    return {"answer": "I could not find that order. Please check the order id."}

def ask_with_tools(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        if ONLINE:
            r = genai_helpers.client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
            m = r.choices[0].message
            if not m.tool_calls:
                return m.content
            messages.append(m)
            for call in m.tool_calls:
                out = run_tool(call.function.name, call.function.arguments)
                print(f"step {step + 1}: {call.function.name}({call.function.arguments}) -> {out}")
                messages.append({"role": "tool", "tool_call_id": call.id, "content": out})
        else:
            d = offline_tool_model(messages)
            if "answer" in d:
                return d["answer"]
            out = run_tool(d["tool"], d["args"])
            print(f"step {step + 1}: {d['tool']}({d['args']}) -> {out}")
            messages.append({"role": "tool", "content": out})
    return "Stopped: too many steps"

print("ANSWER:", ask_with_tools("ORD-1042 arrived late. What refund do I get?"))
print("ANSWER:", ask_with_tools("Where is order ORD-9999?"))

## 6. NEW (v4): LangChain v1 `create_agent` and middleware
The same tools, now as LangChain tools. Middleware adds logging, an audit trail and a call limit without touching the prompt.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, before_model, wrap_tool_call
from langchain_core.tools import tool

@tool
def get_order_status(order_id: str) -> str:
    """Look up an order by id, for example ORD-1042. Returns status, amount and days late."""
    return run_tool("get_order_status", json.dumps({"order_id": order_id}))

@tool
def calculate_refund(amount: float, days_late: int) -> str:
    """Calculate a late delivery refund from the order amount and days late."""
    return run_tool("calculate_refund", json.dumps({"amount": amount, "days_late": days_late}))

AUDIT = []

@before_model
def log_model_call(state, runtime):
    print(f"  [log] model call with {len(state['messages'])} message(s)")

@wrap_tool_call
def audit_tool(request, handler):
    result = handler(request)                       # runs the tool
    AUDIT.append({"tool": request.tool_call["name"], "args": request.tool_call["args"], "result": str(result.content)[:60]})
    print(f"  [audit] {request.tool_call['name']} {request.tool_call['args']}")
    return result

agent = create_agent(get_chat_model(), tools=[get_order_status, calculate_refund],
                     system_prompt="You are a support assistant. Use tools; answer in one sentence.",
                     middleware=[log_model_call, audit_tool, ModelCallLimitMiddleware(run_limit=6)])

for q in ["ORD-1042 arrived 3 days late. What refund do I get?", "Where is ORD-2077?"]:
    out = agent.invoke({"messages": [{"role": "user", "content": q}]})
    print("FINAL:", out["messages"][-1].content, "\n")
print("audit entries:", len(AUDIT))

**Reading older code:** `LLMChain`, `AgentExecutor` and `initialize_agent` are legacy in LangChain v1. Port them to LCEL chains (section 2) or `create_agent` (above).

## 7. NEW (v4): Tool design and tool token cost
Tool definitions are input tokens on every model call. Prices are **placeholders**.

In [ ]:
lean = TOOLS
verbose = [{"type": "function", "function": {"name": f"tool_{i}",
            "description": "This tool can be used in many situations to help the user with a wide variety of "
                           "account, order, billing, network and general questions. " * 3,
            "parameters": {"type": "object", "properties": {f"field_{j}": {"type": "string", "description": "A value"}
                                                            for j in range(5)}}}} for i in range(12)]
PRICE_PER_1M = {"input": 1.00, "output": 4.00}     # PLACEHOLDER prices

def tool_tokens(tools):
    return count_tokens(json.dumps(tools))

for name, tools in [("lean (2 tools)", lean), ("verbose (12 tools)", verbose)]:
    t = tool_tokens(tools)
    per_1000 = t * 4 * 1000 * PRICE_PER_1M["input"] / 1_000_000      # 4 model calls per conversation
    print(f"{name:<20} {t:>5} tokens per call | {per_1000:.2f} per 1,000 conversations (placeholder units)")

**TODO:** rewrite one verbose tool into a lean one (clear name, one-line description with an example). Measure the saving.

## 8. NEW (v4): The OpenAI Agents SDK in the comparison
The same assistant with `Agent`, `function_tool` and `Runner`. Needs an API key; offline we show what would run.

In [ ]:
from agents import Agent, Runner, function_tool

@function_tool
def order_status(order_id: str) -> str:
    """Look up an order by id, for example ORD-1042."""
    return get_order_status_fn(order_id)

@function_tool
def refund_amount(amount: float, days_late: int) -> str:
    """Calculate a late-delivery refund."""
    return calculate_refund_fn(amount, days_late)

support = Agent(name="Support", instructions="Answer order and refund questions with the tools. Be concise.",
                tools=[order_status, refund_amount], model=MODEL)
print("Agent:", support.name, "| tools:", [t.name for t in support.tools])
print("order_status schema:", order_status.params_json_schema)

if ONLINE:
    result = await Runner.run(support, "ORD-1042 arrived 3 days late. What refund do I get?")
    print("FINAL:", result.final_output)
else:
    print("Offline: Runner.run needs an API key. The direct loop (section 5) and create_agent (section 6) show the same steps.")

In [ ]:
import pandas as pd
print(pd.DataFrame([
    {"option": "Direct OpenAI API", "you write": "loop, retries, tracing", "best for": "full control, few layers"},
    {"option": "LangChain v1 + LangGraph", "you write": "tools, middleware, graph", "best for": "workflows, approvals, many integrations"},
    {"option": "OpenAI Agents SDK", "you write": "agents, tools, handoffs", "best for": "fast agent builds with built-in tracing"},
]).to_string(index=False))

## 9. A LangGraph workflow with branches and a retry loop

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

POLICY = {"refund": "Late deliveries get 5% of the order value per day late, up to 50%.",
          "return": "Items can be returned within 30 days in original packaging."}

class State(TypedDict, total=False):
    question: str
    intent: str
    evidence: str
    answer: str
    attempts: int
    path: list

def classify(s):
    q = s["question"].lower()
    intent = "order" if re.search(r"ord-\d{4}", q) else "policy" if any(w in q for w in ["refund", "return", "policy", "late"]) else "other"
    return {"intent": intent, "attempts": s.get("attempts", 0), "path": s.get("path", []) + [f"classify:{intent}"]}

def lookup_order(s):
    oid = re.search(r"ORD-\d{4}", s["question"], re.I).group(0).upper()
    return {"evidence": get_order_status_fn(oid), "path": s["path"] + ["lookup_order"]}

def retrieve_policy(s):
    q = s["question"].lower()
    return {"evidence": " ".join(v for k, v in POLICY.items() if k in q), "path": s["path"] + ["retrieve_policy"]}

def answer(s):
    if not s.get("evidence"):
        return {"answer": "", "path": s["path"] + ["answer:empty"]}
    a = llm(f"Context: {s['evidence']}\nQuestion: {s['question']}", system="Answer only from the context in one sentence.")
    return {"answer": a, "path": s["path"] + ["answer"]}

def check(s):
    return {"attempts": s["attempts"] + 1, "path": s["path"] + ["check"]}

def rewrite(s):
    return {"question": s["question"] + " (refund policy)", "path": s["path"] + ["rewrite"]}

def human(s):
    return {"answer": "Handed to a human agent.", "path": s["path"] + ["human"]}

def route_intent(s):
    return {"order": "lookup_order", "policy": "retrieve_policy"}.get(s["intent"], "human")

def route_check(s):
    if s.get("answer") and "don't know" not in s["answer"].lower():
        return "done"
    return "retry" if s["attempts"] < 2 else "human"

g = StateGraph(State)
for name, fn in [("classify", classify), ("lookup_order", lookup_order), ("retrieve_policy", retrieve_policy),
                 ("answer", answer), ("check", check), ("rewrite", rewrite), ("human", human)]:
    g.add_node(name, fn)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route_intent, {"lookup_order": "lookup_order", "retrieve_policy": "retrieve_policy", "human": "human"})
g.add_edge("lookup_order", "answer")
g.add_edge("retrieve_policy", "answer")
g.add_edge("answer", "check")
g.add_conditional_edges("check", route_check, {"done": END, "retry": "rewrite", "human": "human"})
g.add_edge("rewrite", "retrieve_policy")
g.add_edge("human", END)
app = g.compile()

for q in ["What is the status of ORD-2077?", "How much do I get if delivery is late?", "Can you sing a song?"]:
    out = app.invoke({"question": q})
    print(f"Q: {q}\n   path: {' -> '.join(out['path'])}\n   answer: {out['answer']}\n")

In [ ]:
print(app.get_graph().draw_mermaid())

## 10. NEW (v4): Durable execution, checkpoints and human-in-the-loop interrupts
The refund graph pauses before paying. State is saved per `thread_id`, so the approval can arrive later.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class RefundState(TypedDict, total=False):
    order_id: str
    refund: float
    approved: bool
    result: str
    path: list

def lookup(s):
    o = json.loads(get_order_status_fn(s["order_id"]))
    refund = json.loads(calculate_refund_fn(o["amount"], o["days_late"]))["refund"]
    return {"refund": refund, "path": s.get("path", []) + ["lookup"]}

def approve(s):
    decision = interrupt({"action": "issue_refund", "order_id": s["order_id"], "amount": s["refund"]})
    return {"approved": str(decision).lower() == "yes", "path": s["path"] + [f"approve:{decision}"]}  # default no

def pay(s):
    msg = f"refund of {s['refund']} issued" if s["approved"] else "refund rejected"
    return {"result": msg, "path": s["path"] + ["pay"]}

rg = StateGraph(RefundState)
for n, f in [("lookup", lookup), ("approve", approve), ("pay", pay)]:
    rg.add_node(n, f)
rg.add_edge(START, "lookup"); rg.add_edge("lookup", "approve"); rg.add_edge("approve", "pay"); rg.add_edge("pay", END)
refund_app = rg.compile(checkpointer=InMemorySaver())

cfg_a = {"configurable": {"thread_id": "thread-A"}}
paused = refund_app.invoke({"order_id": "ORD-1042"}, cfg_a)
print("paused at:", refund_app.get_state(cfg_a).next, "| pending:", paused["__interrupt__"][0].value)
print("checkpoints so far:", len(list(refund_app.get_state_history(cfg_a))))

done = refund_app.invoke(Command(resume="no"), cfg_a)
print("thread A ->", done["result"], "|", " -> ".join(done["path"]))

cfg_b = {"configurable": {"thread_id": "thread-B"}}
refund_app.invoke({"order_id": "ORD-1042"}, cfg_b)
print("thread B ->", refund_app.invoke(Command(resume="yes"), cfg_b)["result"])

### Resume after a crash
If a step fails, invoke again on the same thread. Finished steps are not rerun.

In [ ]:
FLAKY = {"fail": True}

def flaky_pay(s):
    if FLAKY["fail"]:
        FLAKY["fail"] = False
        raise RuntimeError("payment service timeout")
    return {"result": "refund issued after retry", "path": s["path"] + ["pay"]}

fg = StateGraph(RefundState)
fg.add_node("lookup", lookup); fg.add_node("pay", flaky_pay)
fg.add_edge(START, "lookup"); fg.add_edge("lookup", "pay"); fg.add_edge("pay", END)
flaky_app = fg.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "thread-C"}}
try:
    flaky_app.invoke({"order_id": "ORD-1042"}, cfg)
except RuntimeError as e:
    print("first run failed:", e, "| saved next step:", flaky_app.get_state(cfg).next)
print("resume:", flaky_app.invoke(None, cfg)["result"])

### The same approval with `HumanInTheLoopMiddleware`
`create_agent` can pause before a risky tool with one line of middleware.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

@tool
def issue_refund(order_id: str, amount: float) -> str:
    """Issue a refund. Risky: a human must approve."""
    return json.dumps({"refunded": amount, "order_id": order_id})

hitl_agent = create_agent(get_chat_model(), tools=[get_order_status, calculate_refund, issue_refund],
                          system_prompt="You are a support assistant.",
                          middleware=[HumanInTheLoopMiddleware(interrupt_on={"issue_refund": True})],
                          checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "agent-1"}}
out = hitl_agent.invoke({"messages": [{"role": "user", "content": "ORD-1042 was 3 days late. Please issue my refund."}]}, cfg)
if "__interrupt__" in out:
    req = out["__interrupt__"][0].value["action_requests"][0]
    print("needs approval:", req["name"], req["args"])
    out = hitl_agent.invoke(Command(resume={"decisions": [{"type": "reject", "message": "Supervisor review needed."}]}), cfg)
print("FINAL:", out["messages"][-1].content)

### TODO 1: Add a `search_policy(query)` tool to the agent and measure its schema tokens.
### TODO 2 (SmartDesk step 22): make `issue_credit` a risky tool that pauses for approval, and log every decision.

In [ ]:
# TODO: write your code here

## 11. Check yourself
1. Where does argument validation happen in each of the three options?
2. How many tokens did the verbose tools add per call?
3. What would be lost if the refund graph used no checkpointer?

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)